# Setup

## Imports

In [1]:
import geopandas as gpd
import os
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import scipy.stats as stats

from utils import CensusDataLoader, ERSDataLoader

os.chdir('/home/jovyan/JupyterRoot/Projects/RHIP')
load_dotenv()

/home/jovyan/JupyterRoot/Projects/RHIP/utils/acs_loader.py:10: UserWarning: Mapping functions unavailable due to import error: NameError. To use mapping features, ensure all dependencies are properly installed: pip install pytidycensus[map]
  import pytidycensus as tc


True

## Config

In [2]:
YEARS = [2013, 2018, 2023]

ACS_CACHE_PATH = Path(".data/RSS_ACS_DATA.csv")
ICE_DATA_DIR = Path("ice-detention-trends/facilities/by_fiscal_year")
METADATA_DIR = Path("ice-detention-trends/metadata")

# Data Collection

In [3]:
acs_loader = CensusDataLoader(api_key=os.getenv("CENSUS_API_KEY"))
ers_loader = ERSDataLoader(continental=True)

Census API key has been set for this session.
Getting data from the 2018-2022 5-year ACS


## RUCC Data

In [4]:
rucc_data = ers_loader.collect_rucc_data()
rucc_data["GEOID"] = rucc_data["GEOID"].astype(int)
rucc_data.head()

#TODO: add matplotlib histogram for RUCC column to visualize distribution

Attribute,GEOID,State,County_Name,Description,Population_2020,RUCC
0,1001,AL,autauga,"Metro - Counties in metro areas of 250,000 to ...",58805,2
1,1003,AL,baldwin,Metro - Counties in metro areas of fewer than ...,231767,3
2,1005,AL,barbour,"Nonmetro - Urban population of 5,000 to 20,000...",25223,6
3,1007,AL,bibb,Metro - Counties in metro areas of 1 million p...,22293,1
4,1009,AL,blount,Metro - Counties in metro areas of 1 million p...,59134,1


## County Shapefile Data

Since the `Pygris` library the collect_geometry_data function uses is basically a wrapper for getting Census Bureau Shape Files, the only columns needed from this dataframe are `GEOID` and `geometry` which provides the polygon objects that geopandas uses for spatial analylsis.

In [5]:
if "county_geo" not in locals():
    county_geo = acs_loader.collect_geometry_data(geometry_type="counties", year=2021)
    
county_geo.head()

,GEOID,geometry
0,10005,"POLYGON ((-75.7226 38.82986, -75.61542 38.8336..."
1,10001,"POLYGON ((-75.7601 39.29682, -75.75 39.29747, ..."
2,10003,"MULTIPOLYGON (((-75.56555 39.51485, -75.56174 ..."
3,2090,"POLYGON ((-148.66326 64.59079, -148.64821 64.5..."
4,2275,"MULTIPOLYGON (((-132.0696 56.03168, -132.06865..."


## Vira Data


### Facilities

Facilities are processed by first reading in the csv from the Vira data directory. The latitude and longitude points are used to convert `facilities` to a geodataframe. 

In [6]:
if "facilities" not in locals():
    # Load facility data   
    facilities = pd.read_csv(METADATA_DIR / "facilities.csv")
    facilities = gpd.GeoDataFrame(
        facilities,
        geometry=gpd.points_from_xy(facilities.longitude, facilities.latitude),
        crs="EPSG:4326",
    )


    facilities = gpd.sjoin(
        facilities,
        county_geo[["GEOID", "geometry"]],
        how="left",
        predicate="within",
    )

    facilities = facilities.drop_duplicates(subset="detention_facility_code")

    n_matched = facilities.GEOID.notna().sum()
    print(f"  Facilities with county match: {n_matched}/{len(facilities)} ({round(n_matched / len(facilities) * 100, 2)})")

    facilities.dropna(subset="GEOID", inplace=True)

    cols = ['address', 'city', 'latitude', 'longitude',
       'county', 'state', 'zip', 'aor', 'index_right']
    facilities.drop(columns=cols, inplace=True)
    facilities.rename({'geometry': "coordinates"}, inplace=True)

#TODO: add matplotlib histogram for type_detailed and type_grouped columns to visualize distribution. should be a 1x2 matplotlib fig object

facilities.head()

  Facilities with county match: 1473/1490 (98.86)


,detention_facility_code,detention_facility_name,type_detailed,type_grouped,geometry,GEOID
0,AAORDMD,Ordnance Road Correctional Center,IGSA,Non-Dedicated,POINT (-76.59416 39.19961),24003.0
1,ABIRJVA,B.R.R.J. Abingdon,Unknown,Other/Unknown,POINT (-81.91164 36.73244),51191.0
2,ABQHOLD,Albuquerque Hold Room,Hold,Hold/Staging,POINT (-106.62703 35.05226),35001.0
3,ABRMCTX,Abilene Regional Med Ctr,Hospital,Medical,POINT (-99.74385 32.37501),48441.0
4,ABRXSPA,Abraxas Academy Detention Center,Juvenile,Family/Youth,POINT (-75.91592 40.19178),42011.0


### Incarceration Data

In [7]:
if "detention_data" not in locals():
    print("Processing Vira data")
    dfs = []
    for yr in YEARS:
        fp = ICE_DATA_DIR / f"FY{yr}.csv"
        if not fp.exists():
            print(f"  WARN: {fp} not found, skipping")
            continue
        df = pd.read_csv(fp)
        df["fiscal_year"] = yr
        dfs.append(df)
    
    detention_data = pd.concat(dfs, ignore_index=True)
    
    # Group by facility and year (some facilities have multiple rows per year)
    detention_data = detention_data.groupby(
        ["detention_facility_code", "fiscal_year"]
    ).agg({"daily_pop": "sum", "midnight_pop": "sum"}).reset_index()
    

    detention_data = detention_data.merge(
    facilities
        ,
        on="detention_facility_code",
        how="left",
    )

else:
    print("Vira data already loaded")

detention_data.head()

Processing Vira data


,detention_facility_code,fiscal_year,daily_pop,midnight_pop,detention_facility_name,type_detailed,type_grouped,geometry,GEOID
0,AAORDMD,2013,0,0,Ordnance Road Correctional Center,IGSA,Non-Dedicated,POINT (-76.59416 39.19961),24003.0
1,AAORDMD,2018,27195,26818,Ordnance Road Correctional Center,IGSA,Non-Dedicated,POINT (-76.59416 39.19961),24003.0
2,AAORDMD,2023,0,0,Ordnance Road Correctional Center,IGSA,Non-Dedicated,POINT (-76.59416 39.19961),24003.0
3,ABIRJVA,2013,0,0,B.R.R.J. Abingdon,Unknown,Other/Unknown,POINT (-81.91164 36.73244),51191.0
4,ABIRJVA,2018,0,0,B.R.R.J. Abingdon,Unknown,Other/Unknown,POINT (-81.91164 36.73244),51191.0


## ACS Data

### Variables

- `MOBILITY_VARS`: Variables for calculating the percentage of the population below the poverty line that moved in the last year as a measure of housing stability in an area.

- `SNAP_VARS`: Variables for calculating the percentage of households that recieve SNAP benefits out of the total number of households.

- `EMPLOYMENT_VARS`: Variables for calculating the percentage of people considered to be in the labor force that are unemployed out of the total number of people in the labor force

- `EDUCATION_VARS`: Variables for calculating the percentage of people over 25 (honestly, no idea why this is the age minimum the Census Bureau uses) without a high school degree.

- `QUALITY_VARS`: Variables for calculating the percentage of occupied housing units with an incomplete kitchen or incomplete plumbing out of the total number of housing units

- `POVERTY_VARS`: Variables for calculating the poverty rate of an area

- `MORTGAGE_VARS`/`RENT_VARS`: variables for calculating the percentage of households in an area that spend more than 30% of their income on housing.

- `MISC_VARS`: Just needed to sneak the household Gini index variable in somewhere

In [8]:
MOBILITY_VARS = {
    "MOBILITY_TOTAL": "B07012_002E",
    # NOTE: `SAME_HOUSE` isn't needed because only want people that moved factored into percentage
    "MOVED_SAME_COUNTY": "B07012_010E",
    "MOVED_DIFF_COUNTY": "B07012_014E",
    "MOVED_DIFF_STATE": "B07012_018E",
    "MOVED_ABROAD": "B07012_022E",
}
SNAP_VARS = {"SNAP_TOTAL": "B22003_001E", "RECEIVING": "B22003_002E"}
EMPLOYMENT_VARS = {"LABOR_FORCE": "B23025_002E", "UNEMPLOYED": "B23025_005E"}
EDUCATION_VARS = {
    "EDU_TOTAL": "B15003_001E",
    "GRADES": [f"B15003_{num:03d}E" for num in range(2, 17)],
}
QUALITY_VARS = { 
    "KITCH_TOTAL": "B25051_001E",
    "LACK_KITCHEN": "B25051_003E",
}
POVERTY_VARS = {"POV_TOTAL": "B17001_001E", "POVERTY": "B17001_002E"}
MORTGAGE_VARS = {
    "MORT_TOTAL": "B25091_001E",
    "MORT_30_35": "B25091_008E",
    "MORT_35_40": "B25091_009E",
    "MORT_40_50": "B25091_010E",
    "MORT_50_PLUS": "B25091_011E",
    "NO_MORT_30_35": "B25091_019E",
    "NO_MORT_35_40": "B25091_020E",
    "NO_MORT_40_50": "B25091_021E",
    "NO_MORT_50_PLUS": "B25091_022E",
}
RENT_VARS = {
    "RENT_TOTAL": "B25070_001E",
    "RENT_30_35": "B25070_007E",
    "RENT_35_40": "B25070_008E",
    "RENT_40_50": "B25070_009E",
    "RENT_50_PLUS": "B25070_010E",
}
MISC_VARS = {"GINI": "B19083_001E"}

In [9]:
# Flatten all var dicts into one lookup
all_vars_dict: dict[str, str] = {}
for d in [
    MOBILITY_VARS, SNAP_VARS, EMPLOYMENT_VARS, EDUCATION_VARS,
    QUALITY_VARS, POVERTY_VARS, MORTGAGE_VARS, RENT_VARS, MISC_VARS,
]:
    for key, val in d.items():
        if isinstance(val, list):
            for i, item in enumerate(val):
                all_vars_dict[f"{key}_{i}"] = item
        else:
            all_vars_dict[key] = val

In [ ]:
CACHE: bool = False # set to false to ignore existing csv file for data
if ACS_CACHE_PATH.exists() and CACHE is True:
    print("Loading cached ACS data from", ACS_CACHE_PATH)
    acs_data = pd.read_csv(ACS_CACHE_PATH)
else:
    print("Fetching ACS data from Census API...")
    acs_data = acs_loader.fetch_multiple_years(
        years=YEARS,
        variables=all_vars_dict,
        geography="county",
    )
    ACS_CACHE_PATH.parent.mkdir(exist_ok=True)
    acs_data.to_csv(ACS_CACHE_PATH, index=False)
    print(f"Saved ACS data to {ACS_CACHE_PATH}")

acs_data.head()

Fetching ACS data from Census API...
Saved ACS data to .data/RSS_ACS_DATA.csv
['GEOID', 'MOBILITY_TOTAL', 'MOVED_SAME_COUNTY', 'MOVED_DIFF_COUNTY', 'MOVED_DIFF_STATE', 'MOVED_ABROAD', 'SNAP_TOTAL', 'RECEIVING', 'LABOR_FORCE', 'UNEMPLOYED', 'EDU_TOTAL', 'GRADES_0', 'GRADES_1', 'GRADES_2', 'GRADES_3', 'GRADES_4', 'GRADES_5', 'GRADES_6', 'GRADES_7', 'GRADES_8', 'GRADES_9', 'GRADES_10', 'GRADES_11', 'GRADES_12', 'GRADES_13', 'state', 'county', 'NAME', 'GRADES_14', 'KITCH_TOTAL', 'LACK_KITCHEN', 'POV_TOTAL', 'POVERTY', 'MORT_TOTAL', 'MORT_30_35', 'MORT_35_40', 'MORT_40_50', 'MORT_50_PLUS', 'NO_MORT_30_35', 'NO_MORT_35_40', 'NO_MORT_40_50', 'NO_MORT_50_PLUS', 'RENT_TOTAL', 'RENT_30_35', 'RENT_35_40', 'RENT_40_50', 'RENT_50_PLUS', 'GINI', 'year']


In [11]:
acs_data["RiskyMobility"] = (
    acs_data["MOVED_DIFF_COUNTY"] + acs_data["MOVED_DIFF_STATE"]
    + acs_data["MOVED_ABROAD"]
) / acs_data["MOBILITY_TOTAL"].replace(0, np.nan)

acs_data["SnapRate"] = acs_data["RECEIVING"] / acs_data["SNAP_TOTAL"].replace(0, np.nan)

acs_data["Unemployment"] = (
    acs_data["UNEMPLOYED"] / acs_data["LABOR_FORCE"].replace(0, np.nan)
)

hs_grades = [
    "GRADES_0", "GRADES_1", "GRADES_2", "GRADES_3", "GRADES_4",
    "GRADES_5", "GRADES_6", "GRADES_7", "GRADES_8", "GRADES_9",
    "GRADES_10", "GRADES_11", "GRADES_12",
]
acs_data["No_HS"] = (
    acs_data[hs_grades].sum(axis=1)
    / acs_data["EDU_TOTAL"].replace(0, np.nan)
)

acs_data["HousingQuality"] = (
    acs_data["LACK_KITCHEN"]
    / acs_data["KITCH_TOTAL"]).replace(0, np.nan)


acs_data["Poverty"] = (
    acs_data["POVERTY"] / acs_data["POV_TOTAL"].replace(0, np.nan)
)

cost_cols = [
    "MORT_30_35", "MORT_35_40", "MORT_40_50", "MORT_50_PLUS",
    "NO_MORT_30_35", "NO_MORT_35_40", "NO_MORT_40_50", "NO_MORT_50_PLUS",
    "RENT_30_35", "RENT_35_40", "RENT_40_50", "RENT_50_PLUS",
]
acs_data["CostBurden"] = (
    acs_data[cost_cols].sum(axis=1)
    / (acs_data["MORT_TOTAL"] + acs_data["RENT_TOTAL"]).replace(0, np.nan)
)


In [14]:
drop_cols = ['MOBILITY_TOTAL', 'MOVED_SAME_COUNTY', 'MOVED_DIFF_COUNTY', 'MOVED_DIFF_STATE', 'MOVED_ABROAD', 'SNAP_TOTAL', 'RECEIVING', 'LABOR_FORCE', 'UNEMPLOYED', 'EDU_TOTAL', 'GRADES_0', 'GRADES_1', 'GRADES_2', 'GRADES_3', 'GRADES_4', 'GRADES_5', 'GRADES_6', 'GRADES_7', 'GRADES_8', 'GRADES_9', 'GRADES_10', 'GRADES_11', 'GRADES_12', 'GRADES_13', 'GRADES_14', 'KITCH_TOTAL', 'LACK_KITCHEN', 'POV_TOTAL', 'POVERTY', 'MORT_TOTAL', 'MORT_30_35', 'MORT_35_40', 'MORT_40_50', 'MORT_50_PLUS', 'NO_MORT_30_35', 'NO_MORT_35_40', 'NO_MORT_40_50', 'NO_MORT_50_PLUS', 'RENT_TOTAL', 'RENT_30_35', 'RENT_35_40', 'RENT_40_50', 'RENT_50_PLUS']

In [15]:
acs_data.drop(columns=drop_cols)

,GEOID,state,county,NAME,GINI,year,RiskyMobility,SnapRate,Unemployment,No_HS,HousingQuality,Poverty,CostBurden
0,9001,09,001,"Fairfield County, Connecticut",0.5391,2013,0.069576,0.081355,0.098758,0.079914,0.016838,0.091351,0.431633
1,9003,09,003,"Hartford County, Connecticut",0.4682,2013,0.073602,0.130030,0.103014,0.083345,0.014828,0.115941,0.372646
2,9005,09,005,"Litchfield County, Connecticut",0.4413,2013,0.076587,0.062346,0.082902,0.055621,0.018835,0.064364,0.374391
3,9007,09,007,"Middlesex County, Connecticut",0.4333,2013,0.158785,0.057982,0.069947,0.039747,0.009301,0.063748,0.353895
4,9009,09,009,"New Haven County, Connecticut",0.4625,2013,0.073192,0.130940,0.105243,0.081298,0.018799,0.123891,0.430483
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,56037,56,037,"Sweetwater County, Wyoming",0.4411,2023,0.124428,0.075431,0.057732,0.041915,0.017372,0.134393,0.213426
19,56039,56,039,"Teton County, Wyoming",0.5222,2023,0.200123,0.011723,0.024509,0.028404,0.043250,0.070282,0.289247
20,56041,56,041,"Uinta County, Wyoming",0.3647,2023,0.059361,0.059634,0.035063,0.035183,0.033266,0.075674,0.154700
21,56043,56,043,"Washakie County, Wyoming",0.4231,2023,0.157651,0.049751,0.020321,0.049991,0.049610,0.086282,0.230612
